In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import plotly.graph_objects as go
from sqlalchemy import text
from plotly.subplots import make_subplots
import numpy as np
import plotly.express as px

# Connection Credentials
DB_USER = "*****"
DB_PASS = "*****"
DB_HOST = "localhost"
DB_PORT = "****"
DB_NAME = "postgres"

DB_URI = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Initialize Engine
engine = create_engine(DB_URI)



In [2]:
# -- Signal Smoothing: 7-Day Rolling Moving Average to identify true epidemiological waves

sql_query = text("""
SELECT 
    location,
    date,
    new_cases,
    ROUND(AVG(new_cases) OVER (
        PARTITION BY location
        ORDER BY date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    )::numeric, 2) AS seven_day_avg,
    LAG(new_cases, 7) OVER (
        PARTITION BY location
        ORDER BY date
    ) AS cases_last_week
FROM public.coviddeaths
WHERE continent IS NOT NULL
ORDER BY location, date;

""")

# 2. Execute Query directly from PostgreSQL into Pandas (Single Execution with Date Parsing)
print("Executing SQL query against PostgreSQL...")
with engine.connect() as conn:
    df = pd.read_sql_query(sql_query, conn, parse_dates=['date'])

print(f"Query executed successfully! Loaded {len(df):,} records.")

# 3. Data Preparation for Selected Location
country_name = 'Germany'
country_df = df[df['location'] == country_name].copy()

# Calculate Week-over-Week percentage velocity on smoothed figures
country_df['seven_day_avg_lag7'] = country_df['seven_day_avg'].shift(7)


# Use np.nan instead of pd.NA to prevent Plotly orjson TypeError during render
country_df['wow_growth_pct'] = (
    (country_df['seven_day_avg'] - country_df['seven_day_avg_lag7']) 
    / country_df['seven_day_avg_lag7'].replace(0, np.nan)
).astype(float) * 100

# Color coding: Red for acceleration (>=0%), Green for deceleration (<0%)
colors = [
    '#d1d5db' if pd.isna(val) 
    else '#ef4444' if val >= 0 
    else '#22c55e' 
    for val in country_df['wow_growth_pct']
]

# 4. Create 2-Panel Subplot Layout
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        "<b>Epidemiological Signal Smoothing: Daily Cases vs. 7-Day Average</b>",
        "<b>Epidemiological Momentum: Week-over-Week Growth Rate (%)</b>"
    ),
    row_heights=[0.60, 0.40]
)

# TOP PANEL: Raw Cases + 7-Day Moving Average
fig.add_trace(
    go.Bar(
        x=country_df['date'],
        y=country_df['new_cases'],
        name='Daily Reported (Raw Noise)',
        marker_color='#94a3b8',
        opacity=0.40
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=country_df['date'],
        y=country_df['seven_day_avg'],
        name='7-Day Rolling Average',
        line=dict(color='#dc2626', width=2.5)
    ),
    row=1, col=1
)

# BOTTOM PANEL: Week-over-Week Growth Rate (%)
fig.add_trace(
    go.Bar(
        x=country_df['date'],
        y=country_df['wow_growth_pct'],
        name='WoW Growth %',
        marker_color=colors,
        opacity=0.75,
        showlegend=False
    ),
    row=2, col=1
)

# 5. Styling & Layout Updates
fig.update_layout(
    title={
        'text': f"<b>{country_name}: COVID-19 Trajectory & Acceleration Dashboard</b>",
        'y': 0.96,
        'x': 0.01,
        'xanchor': 'left'
    },
    template="plotly_white",
    hovermode="x unified",
    showlegend=True,
    legend=dict(orientation="v", yanchor="top", y=1.1, xanchor="right", x=1),
    height=700,
    margin=dict(l=50, r=40, t=100, b=50)
)



Executing SQL query against PostgreSQL...
Query executed successfully! Loaded 81,060 records.


In [3]:
# Case Fatality Rate (CFR) Trajectory & Wave Severity Analysis
# Severity Trends: Comparing Cumulative Macro Baseline vs. Real-Time 30-Day Incident CFR to isolate saturation signatures

cfr_sql_query = text("""
SELECT
    location,
    date,

    -- 1. Cumulative Case Fatality Rate (%) - Macro Baseline
    ROUND(
        (
            total_deaths::numeric
            / NULLIF(total_cases::numeric, 0)
        ) * 100,
        2
    ) AS cumulative_cfr,

    -- 2. Rolling 30-Day Reported Case Fatality Rate (%) - Real-Time Wave Severity
    ROUND(
        (
            SUM(new_deaths) OVER (
                PARTITION BY location
                ORDER BY date
                ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
            )::numeric
            /
            NULLIF(
                SUM(new_cases) OVER (
                    PARTITION BY location
                    ORDER BY date
                    ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
                )::numeric,
                0
            )
        ) * 100,
        2
    ) AS rolling_30d_cfr

FROM public.coviddeaths
WHERE continent IS NOT NULL
  AND total_cases > 0
ORDER BY location, date;
""")


df = pd.read_sql_query(cfr_sql_query, engine, parse_dates=['date'])

# 3. Filter for Target Country & Cast Numeric Types
target_country = 'Germany'  # Change to any country in your dataset
country_df = df[df['location'] == target_country].copy()

# Cast SQL Decimal columns to Python floats for Plotly compatibility
country_df['cumulative_cfr'] = country_df['cumulative_cfr'].astype(float)
country_df['rolling_30d_cfr'] = country_df['rolling_30d_cfr'].astype(float)


# 4. Build Interactive Dual-Trace Plotly Chart
fig = go.Figure()

# Trace 1: Cumulative CFR (Macro Baseline Anchor)
fig.add_trace(
    go.Scatter(
        x=country_df['date'],
        y=country_df['cumulative_cfr'],
        name='Cumulative Lifetime CFR (Macro Baseline)',
        mode='lines',
        line=dict(color='#64748b', width=2, dash='dot'),
        hovertemplate='<b>Lifetime CFR:</b> %{y:.2f}%<extra></extra>'
    )
)

# Trace 2: Rolling 30-Day Incident CFR (Real-Time Wave Severity Signal)
fig.add_trace(
    go.Scatter(
        x=country_df['date'],
        y=country_df['rolling_30d_cfr'],
        name='30-Day Incident CFR (Real-Time Wave Severity)',
        mode='lines',
        line=dict(color='#dc2626', width=2.5),
        hovertemplate='<b>30-Day Incident CFR:</b> %{y:.2f}%<extra></extra>'
    )
)

# 5. Dashboard Layout & Canvas Styling
fig.update_layout(
    title={
        'text': f"<b>{target_country}: Case Fatality Rate (CFR) Trajectory</b><br><sup>Comparing Real-Time 30-Day Wave Severity vs. Lifetime Cumulative Baseline</sup>",
        'y': 0.93,
        'x': 0.01,
        'xanchor': 'left'
    },
    xaxis=dict(
        title="Date",
        rangeslider=dict(visible=False),
        type="date"
    ),
    yaxis=dict(
        title="Case Fatality Rate (%)",
        ticksuffix="%"
    ),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        orientation="v",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0.75
    ),
    height=600,
    margin=dict(l=50, r=40, t=110, b=50)
)

# Render Visualizations
fig.show()


In [4]:
# Did vaccination reduce deaths?
# Direct Impact Test: Broad Pre vs. Post Vaccination Mortality Shift Evaluation

query = """
WITH vaccination_impact AS (
    SELECT
        dea.location,
        dea.new_deaths_per_million, 
        vac.people_vaccinated_per_hundred
    FROM public.coviddeaths dea
    JOIN public.covidvaccinations vac
        ON dea.location = vac.location
        AND dea.date = vac.date
    WHERE dea.continent IS NOT NULL
      AND vac.people_vaccinated_per_hundred IS NOT NULL
)
SELECT
    location,
    ROUND(AVG(CASE WHEN people_vaccinated_per_hundred < 20.0 THEN new_deaths_per_million END)::numeric, 2) AS avg_deaths_low_vaccination,
    ROUND(AVG(CASE WHEN people_vaccinated_per_hundred >= 60.0 THEN new_deaths_per_million END)::numeric, 2) AS avg_deaths_high_vaccination
FROM vaccination_impact
GROUP BY location
ORDER BY avg_deaths_low_vaccination DESC NULLS LAST;
"""

with engine.connect() as conn:
    df = pd.read_sql(text(query), conn)


# 2. Data Preparation for Visualization
# Filter out incomplete rows and select the Top 20 highest initial mortality countries
plot_df = (
    df.replace(0, pd.NA)
    .dropna(
        subset=['avg_deaths_low_vaccination', 'avg_deaths_high_vaccination']
    )
    .head(20)
    .sort_values(
        by='avg_deaths_low_vaccination', ascending=True
    )  # Keep top values at top of chart
)





# 3. Build Interactive Horizontal Grouped Bar Chart
fig = go.Figure()

# Trace 1: Low Vaccination Phase (< 20%)
fig.add_trace(
    go.Bar(
        y=plot_df['location'],
        x=plot_df['avg_deaths_low_vaccination'],
        name='Low Coverage (< 20%)',
        orientation='h',
        marker=dict(color='#dc2626'),  # Crimson Red
        hovertemplate='<b>%{y}</b><br>Low Vac Daily Deaths/1M: <b>%{x:.2f}</b><extra></extra>'
    )
)

# Trace 2: High Vaccination Phase (≥ 60%)
fig.add_trace(
    go.Bar(
        y=plot_df['location'],
        x=plot_df['avg_deaths_high_vaccination'],
        name='High Coverage (≥ 60%)',
        orientation='h',
        marker=dict(color='#2563eb'),  # Royal Blue
        hovertemplate='<b>%{y}</b><br>High Vac Daily Deaths/1M: <b>%{x:.2f}</b><extra></extra>'
    )
)

# 4. Canvas Styling & Layout
fig.update_layout(
    title={
        'text': "<b>Impact of Vaccination Coverage on Daily Per-Capita Mortality</b>",
        'subtitle': {
            'text': "Comparing Average Daily Deaths Per Million: Early Stage (<20%) vs. Mature Coverage (≥60%)",
            'font': dict(size=13, color='#475569')
        },
        'y': 0.93,
        'x': 0.01,
        'xanchor': 'left'
    },
    xaxis=dict(
        title="Average Daily Deaths Per Million",
        showgrid=True,
        gridcolor="#f1f5f9"
    ),
    yaxis=dict(
        title="",
        tickfont=dict(size=12)
    ),
    barmode='group',
    template="plotly_white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.95,
        xanchor="left",
        x=0.70
    ),
    height=700,
    margin=dict(l=140, r=40, t=110, b=50)
)

# Render Chart
fig.show()

In [5]:

## Hospital capacity vs COVID patients
# Resource Optimization: ICU Rate and Relative Hospital Capacity Strain Ranking

query = """
SELECT
    location,
    date,
    hosp_patients,
    hosp_patients_per_million,
    icu_patients,
    icu_patients_per_million,
    new_cases,
    
    -- Cast the entire percentage expression to ::numeric so ROUND(numeric, int) matches
    ROUND(
        ((icu_patients::numeric / NULLIF(hosp_patients, 0)) * 100)::numeric, 
        2
    ) AS icu_acuity_rate_pct,
    
    DENSE_RANK() OVER (
        PARTITION BY date
        ORDER BY hosp_patients_per_million DESC
    ) AS per_capita_hospital_stress_rank

FROM public.coviddeaths
WHERE continent IS NOT NULL
  AND hosp_patients_per_million IS NOT NULL
ORDER BY date, per_capita_hospital_stress_rank;
"""


# 3. Execute query and fetch into DataFrame
with engine.connect() as conn:
    df = pd.read_sql(text(query), conn)







# Pivot dataframe for heatmap representation
pivot_df = df.pivot(
    index="location", columns="date", values="per_capita_hospital_stress_rank"
)

# Calculate reporting coverage percentage for each country
valid_days = pivot_df.notna().sum(axis=1)
coverage_pct = (valid_days / pivot_df.shape[1]) * 100

# Retain only countries with at least 75% reporting coverage
pivot_df = pivot_df.loc[coverage_pct >= 75]

fig = px.imshow(
    pivot_df,
    labels={
        "x": "Date", 
        "y": "Country", 
        "color": "Rank (1 = highest)"
    },
    x=pivot_df.columns,
    y=pivot_df.index,
    color_continuous_scale="Reds_r",  # Darker red = Rank 1 (higher burden)
    title="<b>Daily Hospital Patients per Million: Country Ranking</b>",
    aspect="auto",
)

fig.update_layout(
    height=600, 
    template="plotly_white",
    coloraxis_colorbar=dict(title="Rank<br>(1 = highest)")
)
fig.update_traces(hoverongaps=False)
fig.show()

In [6]:
'''
Query Name: Reported Hospitalization and Mortality
            Across Vaccination-Coverage Bands

Objective:
Compare average hospitalization burden, reported daily mortality,
and aggregate reported deaths-to-cases ratios across vaccination
coverage levels.

Methodology:
Uses PostgreSQL CASE statements to classify country-date
observations into vaccination-coverage bands and FILTER clauses
to calculate the aggregate ratio using matched case-death observations.

Interpretation:
This is a descriptive country-level comparison, not a cohort-based
study or an estimate of causal vaccination effectiveness.
'''

query = """
WITH banded_data AS (

    SELECT
        dea.location,
        dea.date,

        -- Classify observations by vaccination coverage
        CASE
            WHEN vac.people_vaccinated_per_hundred < 10
                THEN 'Below 10%'
            WHEN vac.people_vaccinated_per_hundred < 20
                THEN '10%-<20%'
            WHEN vac.people_vaccinated_per_hundred < 30
                THEN '20%-<30%'
            WHEN vac.people_vaccinated_per_hundred < 40
                THEN '30%-<40%'
            WHEN vac.people_vaccinated_per_hundred < 50
                THEN '40%-<50%'
            WHEN vac.people_vaccinated_per_hundred < 60
                THEN '50%-<60%'
            ELSE '60% or higher'
        END AS vaccination_coverage_band,

        -- Numeric sorting order
        CASE
            WHEN vac.people_vaccinated_per_hundred < 10 THEN 1
            WHEN vac.people_vaccinated_per_hundred < 20 THEN 2
            WHEN vac.people_vaccinated_per_hundred < 30 THEN 3
            WHEN vac.people_vaccinated_per_hundred < 40 THEN 4
            WHEN vac.people_vaccinated_per_hundred < 50 THEN 5
            WHEN vac.people_vaccinated_per_hundred < 60 THEN 6
            ELSE 7
        END AS sort_order,

        dea.hosp_patients_per_million,
        dea.new_deaths_per_million,
        dea.new_deaths,
        dea.new_cases

    FROM public.coviddeaths dea

    JOIN public.covidvaccinations vac
      ON dea.location = vac.location
     AND dea.date = vac.date

    WHERE dea.continent IS NOT NULL
      AND vac.people_vaccinated_per_hundred IS NOT NULL
)

SELECT
    vaccination_coverage_band,

    -- Number of country-date observations
    COUNT(*) AS country_day_observations,

    -- Number of distinct countries
    COUNT(DISTINCT location) AS countries_included,

    -- Average hospitalization burden
    COUNT(hosp_patients_per_million)
        AS hospitalization_observations,

    ROUND(
        AVG(hosp_patients_per_million)::numeric,
        2
    ) AS avg_hospital_patients_per_million,

    -- Average reported daily mortality
    COUNT(new_deaths_per_million)
        AS mortality_observations,

    ROUND(
        AVG(new_deaths_per_million)::numeric,
        2
    ) AS avg_daily_deaths_per_million,

    -- Number of matched case-death observations
    COUNT(*) FILTER (
        WHERE new_deaths IS NOT NULL
          AND new_cases IS NOT NULL
    ) AS paired_case_death_observations,

    -- Aggregate reported deaths-to-cases ratio
    ROUND(
        (
            (SUM(new_deaths) FILTER (
                WHERE new_deaths IS NOT NULL
                  AND new_cases IS NOT NULL
            ))::numeric
            /
            NULLIF(
                (SUM(new_cases) FILTER (
                    WHERE new_deaths IS NOT NULL
                      AND new_cases IS NOT NULL
                ))::numeric,
                0
            )
        ) * 100,
        2
    ) AS aggregate_reported_deaths_to_cases_pct

FROM banded_data

GROUP BY
    sort_order,
    vaccination_coverage_band

ORDER BY sort_order;
"""


with engine.connect() as conn:
    df = pd.read_sql(text(query), conn)


# 3. Type Conversion
numeric_cols = [
    "avg_hospital_patients_per_million",
    "avg_daily_deaths_per_million",
    "aggregate_reported_deaths_to_cases_pct",  
]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

# 4. Multi-Metric Subplot Visualization
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "Avg Hosp Patients / 1M",
        "Avg Daily Deaths / 1M",
        "Reported Deaths-to-Cases Ratio (%)",
    ),
)

# Hosp Patients per Million
fig.add_trace(
    go.Bar(
        x=df["vaccination_coverage_band"],  
        y=df["avg_hospital_patients_per_million"],
        name="Hosp Patients / 1M",
        marker_color="#2563eb",
        hovertemplate="<b>%{x}</b><br>Hosp Patients/1M: <b>%{y:.2f}</b><extra></extra>",
    ),
    row=1,
    col=1,
)

# Daily Deaths per Million
fig.add_trace(
    go.Bar(
        x=df["vaccination_coverage_band"],  
        y=df["avg_daily_deaths_per_million"],
        name="Daily Deaths / 1M",
        marker_color="#dc2626",
        hovertemplate="<b>%{x}</b><br>Daily Deaths/1M: <b>%{y:.2f}</b><extra></extra>",
    ),
    row=1,
    col=2,
)

# Aggregate Reported Deaths-to-Cases Ratio
fig.add_trace(
    go.Bar(
        x=df["vaccination_coverage_band"], 
        y=df["aggregate_reported_deaths_to_cases_pct"], 
        name="Reported Deaths-to-Cases Ratio:",
        marker_color="#059669",
        hovertemplate="<b>%{x}</b><br>Reported Deaths-to-Cases Ratio: <b>%{y:.2f}%</b><extra></extra>",
    ),
    row=1,
    col=3,
)

# 5. Styling Layout
fig.update_layout(
    title_text="<b>Reported Hospitalization and Mortality by Vaccination-Coverage Band</b>",
    template="plotly_white",
    showlegend=False,
    height=450,
    margin=dict(l=40, r=40, t=80, b=80),
)

fig.update_xaxes(tickangle=-30)
fig.show()





